# Consultas avanzadas en Spark SQL (CSV + Parquet)

Este notebook extiende `dataset_ventas_parquet_csv` para cumplir la consigna:

1. Cargar dataset desde CSV y guardarlo también en Parquet.
2. Ejecutar 3 consultas avanzadas:
   - Filtros y orden encadenados.
   - Agregaciones estadísticas.
   - Join entre DataFrames + selectExpr.
3. Usar `explain()` para ver el plan lógico y físico (Catalyst + Tungsten).


In [31]:
# Paso 1: crear sesión Spark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, count, expr

spark = (
    SparkSession.builder
    .appName("DatasetVentasEjemploAvanzado")
    .getOrCreate()
)


In [32]:
# Paso 2: cargar dataset desde CSV (puede ser tu dataset grande)
input_csv_path = r"D:\000_ANALISTA_DATOS\Bootcamp-main\Modulo9\Leccion4\LC2\data\ventas_big.csv"  # ajusta esta ruta a tu caso

ventas_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(input_csv_path)
)

print("Esquema original del dataset:")
ventas_df.printSchema()
ventas_df.show(5)


Esquema original del dataset:
root
 |-- id_venta: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- tienda_id: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- producto_id: integer (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio_unitario: double (nullable = true)
 |-- descuento_pct: integer (nullable = true)
 |-- monto: double (nullable = true)
 |-- metodo_pago: string (nullable = true)
 |-- cliente_id: integer (nullable = true)

+--------+----------+---------+------+---------+-----------+--------+---------------+-------------+--------+-----------+----------+
|id_venta|     fecha|tienda_id|region|categoria|producto_id|cantidad|precio_unitario|descuento_pct|   monto|metodo_pago|cliente_id|
+--------+----------+---------+------+---------+-----------+--------+---------------+-------------+--------+-----------+----------+
|       1|2026-05-10|       43|   Sur| Deportes|       1905|      10| 

In [33]:
# Paso 3: guardar en Parquet y CSV para el ejercicio
base_path = "data/ventas_spark_advanced"

ventas_df.write.mode("overwrite").parquet(base_path + "/parquet")

ventas_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .option("sep", ",") \
    .csv(base_path + "/csv")

parquet_df = spark.read.parquet(base_path + "/parquet")

csv_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(base_path + "/csv")
)

print("Filas en Parquet:", parquet_df.count())
print("Filas en CSV:", csv_df.count())

parquet_df.createOrReplaceTempView("ventas_parquet")
csv_df.createOrReplaceTempView("ventas_csv")


Filas en Parquet: 500000
Filas en CSV: 500000


In [34]:
# Consulta 1: filtros y ordenamientos encadenados
consulta1_df = (
    parquet_df
    .filter(col("categoria") == "Electronica")
    .filter(col("monto") > 500)
    .orderBy(col("fecha").asc(), col("monto").desc())
)

print("Plan de ejecución (CONSULTA 1):")
consulta1_df.explain(mode="extended")  # Catalyst muestra lógicos + físico [web:61][web:62]

print("Resultado de CONSULTA 1:")
consulta1_df.show(10)


Plan de ejecución (CONSULTA 1):
== Parsed Logical Plan ==
'Sort ['fecha ASC NULLS FIRST, 'monto DESC NULLS LAST], true
+- Filter (monto#1252 > cast(500 as double))
   +- Filter (categoria#1247 = Electronica)
      +- Relation [id_venta#1243,fecha#1244,tienda_id#1245,region#1246,categoria#1247,producto_id#1248,cantidad#1249,precio_unitario#1250,descuento_pct#1251,monto#1252,metodo_pago#1253,cliente_id#1254] parquet

== Analyzed Logical Plan ==
id_venta: int, fecha: date, tienda_id: int, region: string, categoria: string, producto_id: int, cantidad: int, precio_unitario: double, descuento_pct: int, monto: double, metodo_pago: string, cliente_id: int
Sort [fecha#1244 ASC NULLS FIRST, monto#1252 DESC NULLS LAST], true
+- Filter (monto#1252 > cast(500 as double))
   +- Filter (categoria#1247 = Electronica)
      +- Relation [id_venta#1243,fecha#1244,tienda_id#1245,region#1246,categoria#1247,producto_id#1248,cantidad#1249,precio_unitario#1250,descuento_pct#1251,monto#1252,metodo_pago#1253,cl

In [35]:
# Consulta 2: agregaciones con funciones estadísticas
consulta2_df = (
    parquet_df
    .groupBy("categoria")
    .agg(
        sum("monto").alias("total_monto"),
        avg("monto").alias("monto_promedio"),
        count("*").alias("num_ventas")
    )
    .orderBy(col("total_monto").desc())
)

print("Plan de ejecución (CONSULTA 2):")
consulta2_df.explain(mode="extended")  # [web:45][web:61]

print("Resultado de CONSULTA 2:")
consulta2_df.show()


Plan de ejecución (CONSULTA 2):
== Parsed Logical Plan ==
'Sort ['total_monto DESC NULLS LAST], true
+- Aggregate [categoria#1247], [categoria#1247, sum(monto#1252) AS total_monto#1354, avg(monto#1252) AS monto_promedio#1355, count(1) AS num_ventas#1356L]
   +- Relation [id_venta#1243,fecha#1244,tienda_id#1245,region#1246,categoria#1247,producto_id#1248,cantidad#1249,precio_unitario#1250,descuento_pct#1251,monto#1252,metodo_pago#1253,cliente_id#1254] parquet

== Analyzed Logical Plan ==
categoria: string, total_monto: double, monto_promedio: double, num_ventas: bigint
Sort [total_monto#1354 DESC NULLS LAST], true
+- Aggregate [categoria#1247], [categoria#1247, sum(monto#1252) AS total_monto#1354, avg(monto#1252) AS monto_promedio#1355, count(1) AS num_ventas#1356L]
   +- Relation [id_venta#1243,fecha#1244,tienda_id#1245,region#1246,categoria#1247,producto_id#1248,cantidad#1249,precio_unitario#1250,descuento_pct#1251,monto#1252,metodo_pago#1253,cliente_id#1254] parquet

== Optimized Log

In [36]:
# Consulta 3: join entre DataFrames + selectExpr

data_categorias = [
    ("Electronica", "Alta"),
    ("Hogar", "Media"),
    ("Deportes", "Media"),
    ("Juguetes", "Baja"),
]
cols_categorias = ["categoria", "prioridad"]

categorias_df = spark.createDataFrame(data_categorias, cols_categorias)
categorias_df.createOrReplaceTempView("dim_categoria")

consulta3_df = (
    parquet_df.alias("v")
    .join(categorias_df.alias("c"), col("v.categoria") == col("c.categoria"), "left")
    .selectExpr(
        "v.id_venta",
        "v.fecha",
        "v.categoria",
        "c.prioridad as prioridad_categoria",
        "v.cantidad",
        "v.monto",
        "monto * cantidad as ingreso_total"
    )
)

print("Plan de ejecución (CONSULTA 3):")
consulta3_df.explain(mode="extended")  # [web:56][web:61]

print("Resultado de CONSULTA 3:")
consulta3_df.show(10)


Plan de ejecución (CONSULTA 3):
== Parsed Logical Plan ==
'Project ['v.id_venta, 'v.fecha, 'v.categoria, 'c.prioridad AS prioridad_categoria#1401, 'v.cantidad, 'v.monto, ('monto * 'cantidad) AS ingreso_total#1402]
+- Join LeftOuter, (categoria#1247 = categoria#1399)
   :- SubqueryAlias v
   :  +- Relation [id_venta#1243,fecha#1244,tienda_id#1245,region#1246,categoria#1247,producto_id#1248,cantidad#1249,precio_unitario#1250,descuento_pct#1251,monto#1252,metodo_pago#1253,cliente_id#1254] parquet
   +- SubqueryAlias c
      +- LogicalRDD [categoria#1399, prioridad#1400], false

== Analyzed Logical Plan ==
id_venta: int, fecha: date, categoria: string, prioridad_categoria: string, cantidad: int, monto: double, ingreso_total: double
Project [id_venta#1243, fecha#1244, categoria#1247, prioridad#1400 AS prioridad_categoria#1401, cantidad#1249, monto#1252, (monto#1252 * cast(cantidad#1249 as double)) AS ingreso_total#1402]
+- Join LeftOuter, (categoria#1247 = categoria#1399)
   :- SubqueryAlia

## Catalyst y Tungsten en estos planes

- **Catalyst** (optimizador lógico) toma las operaciones de alto nivel
  (`filter`, `groupBy`, `join`, `selectExpr`) y:
  - Empuja filtros hacia la fuente de datos (predicate pushdown).
  - Elimina columnas no usadas.
  - Elige estrategias de agregación/join eficientes según el tamaño estimado. [web:61][web:69]

- **Tungsten** ejecuta el plan físico resultante:
  - Usa representación binaria en memoria para cada fila.
  - Administra la memoria de forma explícita y genera código (whole-stage codegen)
    para aprovechar CPU y caché al máximo. [web:64][web:66]


In [37]:
# Cerramos la sesión Spark al terminar
# Siempre es buena práctica liberar los recursos
spark.stop()
print("Sesión Spark cerrada. Notebook completado exitosamente.")

Sesión Spark cerrada. Notebook completado exitosamente.
